# 逐日配置变化与事后平均配置对照

均值使用评价期预测，不可部署；固定三种成本，不优化配置比例。同EMA/HOLD和开盘路由，保留每日分类输出。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _ensemble
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.backtest.risk_cap_ablation import LegalCapOpenPolicy
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/tangent_mean_allocation_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us',kernels=1)
frozen,fa=_source(root,plan['frozen_source'],['analysis/predictions','analysis/targets','models/candidate_71'])
layer,la=_source(root,plan['risk_layer_source'],['inputs/plan','inputs/frame',*[f'analysis/native/{family}_{cost}' for family in ['bypass','full_stack','soxx'] for cost in plan['slippage_bps']]])
controller,ca=_source(root,plan['price_source'],['inputs/controller']);action,aa=_source(root,plan['action_source'],['inputs/plan','inputs/market'])
for key in ['tariff','initial_cash','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert layer['inputs/plan'][key]==plan[key]
frame=layer['inputs/frame'].copy();dates=pd.DatetimeIndex(frame.signal_date);market=action['inputs/market'];market_dates=pd.DatetimeIndex(market.signal_date);model=frozen['models/candidate_71'];raw=_ensemble(frozen['analysis/predictions'],'candidate',plan['seeds']).reindex(market_dates);returns=market.set_index('signal_date')[['soxl_return','soxx_return']]
reconstructed=_smooth_weights(raw,spec=model.spec,candidate=model.candidate,asset_returns=returns);np.testing.assert_allclose(reconstructed,frozen['analysis/targets']['candidate'],atol=1e-10,rtol=1e-10)
weights=['soxl_weight','soxx_weight','cash_weight'];mean=raw.reindex(dates)[weights].mean();assert np.isclose(mean.sum(),1.,atol=1e-12) and mean.soxl_weight<=plan['maximum_soxl_weight']+1e-12 and mean.cash_weight>=plan['minimum_cash_weight']-1e-12
constant=raw.copy()
for column in weights:constant[column]=mean[column]
pd.testing.assert_frame_equal(raw.drop(columns=weights),constant.drop(columns=weights));targets=_smooth_weights(constant,spec=model.spec,candidate=model.candidate,asset_returns=returns).reindex(dates)
for c in targets:frame[c]=targets[c].to_numpy()
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='posthoc_mean_target_control',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'frozen':fa,'layer':la,'controller':ca,'action':aa},raw_predictions=raw,constant_raw_predictions=constant,mean_weights=mean,frame=frame);rec.log_params(new_fits=0,new_native_cases=3,promotion_eligible=False)
books={};audits={};spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec'])
for cost in plan['slippage_bps']:
 policy=LegalCapOpenPolicy(open_model=controller['inputs/controller']['frozen_open_action_value_model'],action_spec=spec,maximum_soxl_weight=plan['maximum_soxl_weight'],minimum_cash_weight=plan['minimum_cash_weight']);replay=replay_full_path(frame,models=[policy],plan=plan,slippage_bps=cost);book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=cost,initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];books[('posthoc_mean',cost)]=book;audits[str(cost)]=pd.DataFrame(policy.audit)
 for f in ['bypass','full_stack','soxx']:books[(f,cost)]=layer[f'analysis/native/{f}_{cost}']
 rec.save_objects(artifact_path='analysis/native',**{f'{f}_{cost}':books[(f,cost)] for f in ['posthoc_mean','bypass','full_stack','soxx']});rec.log_metrics(completed_new_native_cases=len(audits));print('account',cost,flush=True)
rows=[];annual=[];paired=[]
for (family,cost),book in books.items():
 path=book['authority'];benchmark=books[('soxx',cost)]['authority'];assert path.trade_date.equals(benchmark.trade_date)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,benchmark),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'average_soxl_pct':100*path.ending_soxl_weight.mean(),'average_soxx_pct':100*path.ending_soxx_weight.mean(),'average_cash_pct':100*path.ending_cash_weight.mean(),'average_nominal_beta':float((3*path.ending_soxl_weight+path.ending_soxx_weight).mean()),'orders':len(book['orders']),'broker_fees_usd':path.broker_and_external_fees_usd.sum(),'slippage_usd':path.slippage_reserve_usd.sum()});annual.append(annual_paired_metrics(path,benchmark).assign(family=family,slippage_bps=cost))
for cost in plan['slippage_bps']:paired.append({'slippage_bps':cost,**paired_metrics(books[('bypass',cost)]['authority'],books[('posthoc_mean',cost)]['authority'])})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);comparison=pd.DataFrame(paired)
summary={'experiment':card.name,'decision':'posthoc_mean_control_diagnostic_only','objective_achieved':False,'promotion_eligible':False,'new_fits':0,'new_native_cases':3,'reused_native_cases':9,'mean_raw_weights':mean.to_dict(),'raw_mean_nominal_beta':float(3*mean.soxl_weight+mean.soxx_weight),'dynamic_beats_posthoc_mean_all_costs':bool(comparison.relative_wealth.gt(1+1e-8).all()),'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'scope':plan['scope'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/strategy.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,dynamic_vs_mean=comparison,open_decisions=audits,summary=summary);artifacts=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);atomic_json(summary,card/'summary.json');table.to_csv(card/'backtest_benchmark_comparison.csv',index=False);annual.to_csv(card/'annual_metrics.csv',index=False);comparison.to_csv(card/'dynamic_vs_mean.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct','average_nominal_beta']].to_string(index=False));print(mean)


In [1]:
from pathlib import Path
import json,hashlib,numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd();card=root/'research/tangent_mean_allocation_20261014';s=json.loads((card/'summary.json').read_text());qlib.init(provider_uri=s['provider_uri'],region='us',kernels=1)
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=card.name);raw=rec.load_object('inputs/raw_predictions');frame=rec.load_object('inputs/frame');mean=rec.load_object('inputs/mean_weights');dates=pd.DatetimeIndex(frame.signal_date);values=raw.reindex(dates)[['soxl_weight','soxx_weight','cash_weight']];assert len(values)==944 and dates.is_unique and values.notna().all().all();np.testing.assert_allclose(values.mean(),mean,rtol=0,atol=1e-12)
 with R.start(experiment_name=card.name,recorder_id=s['recorder_id'],resume=True):
  rec=R.get_recorder();rec.save_objects(artifact_path='analysis',mean_coverage={'evaluated_signals':944,'unknown_weights':0,'all_evaluation_dates_in_mean':True,'mean_uses_future_evaluation_predictions':True,'promotion_eligible':False});artifacts=Path(s['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');s['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=s['experiment_id']);atomic_json(s,card/'summary.json')


# 逐日配置与事后均值：有小幅历史差异，尚非风险合格超额

0新增拟合，3套新增完整原生Qlib账户，9套既有账户复用。保留已冻结三seed平均模型，固定944个执行区间（2023-01-03信号起至2026-10-09结算），10万美元、原IBKR Tiered费用投影和0/2/5bp独立状态。

事后均值取这944个信号日的原始配置输出，SOXL 25.39062491%、SOXX 41.41852349%、CASH 33.19085161%，不优化比例、不按收益选日期。944行权重全部已知，无缺失跳过。均值使用评价期预测，评价开始时无法获得，明确不可部署、不可晋升。

只替换原始三资产权重，仍经过相同EMA/accepted-HOLD、合法仓位投影及开盘执行路由。每日tail/alpha分类输出保持原样，不是没有任何预测信息的被动组合；开盘路由规则相同，但目标与自身持仓变化会改变它的输入和决策。两组都采用上一卡的动态风险层旁路，合法最大名义beta 1.45不变。

|账户|滑点|累计净收益|监控最大回撤|平均现金|平均名义beta|
|---|---|---|---|---|---|
|逐日配置（无动态风险层）|0bp|+436.91%|-49.82%|33.39%|1.1696|
|事后均值原始配置|0bp|+414.21%|-51.20%|33.12%|1.1777|
|完整小模型策略|0bp|+342.52%|-27.59%|43.27%|0.9098|
|SOXX|0bp|+387.75%|-43.38%|0.15%|0.9985|
|逐日配置（无动态风险层）|2bp|+435.22%|-49.86%|33.39%|1.1696|
|事后均值原始配置|2bp|+413.80%|-51.22%|33.13%|1.1777|
|完整小模型策略|2bp|+335.36%|-27.64%|43.67%|0.8993|
|SOXX|2bp|+387.73%|-43.39%|0.14%|0.9986|
|逐日配置（无动态风险层）|5bp|+432.92%|-49.94%|33.39%|1.1696|
|事后均值原始配置|5bp|+412.94%|-51.24%|33.12%|1.1777|
|完整小模型策略|5bp|+348.06%|-27.73%|43.67%|0.8994|
|SOXX|5bp|+387.70%|-43.39%|0.12%|0.9988|

逐日配置在0/2/5bp下相对事后均值的期末财富差异分别+4.41%/+4.17%/+3.90%。2bp累计净收益差为21.43个百分点，名义暴露略低（1.1696 vs 1.1777），监控回撤也略低（49.86% vs 51.22%）。因此这段路径的全部差异不能简单归为逐日策略平均暴露更高；但平均原始权重相同不等于实现风险完全匹配，也没有证明独立、可重复的预测alpha。

|信号年|逐日配置2bp净收益|事后均值2bp净收益|逐日/均值财富差异|
|---|---|---|---|
|2023|+60.56%|+62.26%|-1.05%|
|2024|+20.17%|+18.24%|+1.63%|
|2025|+53.79%|+48.64%|+3.46%|
|2026|+80.39%|+80.17%|+0.12%|

2bp逐日策略共1209笔订单，均值对照940笔；逐日策略佣金及外部费用约824.80美元、滑点约775.68美元，对照约506.17/328.78美元。正的净差异不是依靠更少交易费用得到，但这仍是事后单条历史路径比较，没有预先登记的独立统计显著性检验。不能把三个相近成本情景当三个独立样本。

名义beta按3×SOXL权重+SOXX权重计算，不是统计回归beta。均值对照保留分类输出和路由，因此其SOXX超额不能全部归为纯杠杆补偿。控制匹配的是原始权重均值，不是每天实际暴露、波动或回撤；不通过事后迭代配置把这些风险指标调到相同。

决定：posthoc_mean_control_diagnostic_only，promotion_eligible=false。两个高暴露账户的回撤都接近或超过50%，不满足30%约束。完整策略风险更低但收益仍落后SOXX；稳定净超额目标未完成，现役不变。保留逐日配置的小幅历史差异，不将诊断均值作为新策略，不继续优化这个均值或杠杆。

下一项问题是如何在风险合格条件下验证、保留配置变化的价值；不能仅以收益超过SOXX就宣称目标达成，也不能仅凭这份事后对照证明择时无效或有效。后续仍需严格历史训练/回测及独立验证，不能改变风险要求。

验证：原始目标平滑结果与父Recorder逐值一致；每日分类输出未改；完整944日均值覆盖、合法权重、原生账户现金/数量/费用对账、共同日历及NAS冷读回通过。本轮未改底层策略代码，不重复训练或添加重复测试。

[完整比较](backtest_benchmark_comparison.csv) · [逐日对均值](dynamic_vs_mean.csv) · [年度](annual_metrics.csv) · [固定计划](plan.json)

Recorder `efdaf2655e834409bd25d2593c6bdab8`。
